# Enterprise Orchestration Lab — GRPO RL Training

This notebook runs **Group Relative Policy Optimization (GRPO)** training using TRL on the Enterprise Orchestration RL Environment.

**Pipeline overview:**
1. Load `Qwen2.5-1.5B-Instruct` with LoRA
2. Sample environment states as prompts
3. Model generates candidate actions (JSON)
4. Each action runs through `env.step()` for **verifiable rewards**
5. GRPO updates the policy to prefer high-reward actions

**Requirements:** Google Colab with a T4 GPU (free tier is sufficient)

---
## 1. Install Dependencies

We install TRL, PEFT, and other training dependencies. Unsloth is attempted but optional.

In [ ]:
# Fix numpy first to prevent version conflicts
!pip install -q --upgrade numpy

# Install core training dependencies (with deps so nothing breaks)
!pip install -q trl peft accelerate bitsandbytes datasets matplotlib

# Attempt unsloth install (optional — will fall back to HF if it fails)
!pip install -q --no-deps unsloth unsloth_zoo 2>/dev/null || echo 'Unsloth install skipped (optional).'

# Install environment dependency
!pip install -q openenv-core

---
## 2. Clone the Environment Repository

In [ ]:
import os, sys, json, re

if not os.path.exists('scaler-final'):
    !git clone https://github.com/redhatsam09/scaler-final.git

os.chdir('scaler-final')
!pip install -q -r requirements.txt
sys.path.insert(0, '.')

print('Repository cloned and dependencies installed.')

---
## 3. Verify Environment Works

Import the core environment classes and confirm everything loads correctly.

In [ ]:
from src.environment import DataCleaningEnv
from src.graders import EnterpriseOrchestrationGrader, MissingValuesGrader
from src.models import Action

env = DataCleaningEnv(seed=42)
obs = env.reset(task_id='task_enterprise_orchestration', seed=42)

print(f'Dataset shape: {obs.dataset_shape[0]} rows x {obs.dataset_shape[1]} cols')
print(f'Available actions: {obs.available_actions}')
print(f'Missing values: {dict(list(obs.missing_values.items())[:5])}')
print(f'KPIs: {json.dumps(obs.kpi_snapshot, indent=2)}')
print(f'\nObservation (first 300 chars):')
print(obs.natural_language_observation[:300])

---
## 4. Define the Environment-Grounded Reward Function

This is the core of our RL pipeline. Each LLM completion is:
1. Parsed from JSON into an `Action` object
2. Executed via `env.step(action)` against the real environment
3. Scored using the environment step reward + the grader rubric

No proxy rewards — every score comes from the actual environment.

In [ ]:
SYSTEM_PROMPT = """You are an enterprise workflow orchestrator managing CRM, Billing, and Support systems.
Given the current state, output a JSON action with:
- action_type: one of [analyze, impute, deduplicate, validate, report_findings, delegate, resolve_alert, reconcile_apps, oversight_review, inspect_actor, audit_records, request_policy_clarification]
- target_columns: list of column names
- parameters: dict of action parameters
- reasoning: why you chose this action (minimum 15 chars)

Respond ONLY with valid JSON."""


def build_prompt(obs):
    """Convert environment observation to an LLM prompt."""
    return f"""{SYSTEM_PROMPT}

Current State:
{obs.natural_language_observation}

Dataset: {obs.dataset_shape[0]} rows, {obs.dataset_shape[1]} cols
Missing values: {json.dumps(dict(list(obs.missing_values.items())[:6]))}
Step: {obs.step_count}
Policy version: {obs.policy_version}
KPIs: {json.dumps(obs.kpi_snapshot)}
Actor messages: {obs.actor_messages[-2:] if obs.actor_messages else []}
Urgency: {obs.urgency_signals if obs.urgency_signals else 'None'}

What action should you take?"""


def environment_reward_function(completions, prompts, **kwargs):
    """
    GRPO reward function: runs each completion through the ACTUAL environment.
    Returns verifiable rewards — no proxy or heuristic.
    """
    rewards = []
    reward_env = DataCleaningEnv(seed=42)
    task_ids = ['task_enterprise_orchestration', 'task_missing_values',
                'task_duplicate_handling', 'task_complex_validation']

    for i, completion in enumerate(completions):
        try:
            task_id = task_ids[i % len(task_ids)]
            obs = reward_env.reset(task_id=task_id, seed=1000 + i)

            # Parse LLM output into Action
            match = re.search(r'\{.*\}', completion, re.DOTALL)
            if match:
                data = json.loads(match.group())
                action = Action(
                    action_type=data.get('action_type', 'analyze'),
                    target_columns=data.get('target_columns', obs.column_names[:3]),
                    parameters=data.get('parameters', {}),
                    reasoning=data.get('reasoning', 'Model-generated'),
                )
            else:
                rewards.append(-0.5)  # Invalid JSON penalty
                continue

            # Run action through REAL environment
            _, reward, _, _ = reward_env.step(action)

            # Get grader score
            grader = EnterpriseOrchestrationGrader if 'enterprise' in task_id else MissingValuesGrader
            grade = grader.grade(reward_env.current_episode)

            # Combined reward: step signal + partial grade + format bonus
            combined = 0.6 * reward.value + 0.4 * float(grade) + 0.1
            rewards.append(min(1.0, max(-1.0, combined)))

        except Exception as e:
            rewards.append(-0.5)

    return rewards


# Quick test: verify the reward function works
test_completion = json.dumps({
    'action_type': 'analyze',
    'target_columns': ['account_id'],
    'parameters': {},
    'reasoning': 'Profile the dataset quality before taking action'
})
test_reward = environment_reward_function([test_completion], ['test'])
print(f'Test reward for analyze action: {test_reward[0]:.4f}')

---
## 5. Load Model with LoRA

We attempt Unsloth for 4-bit optimization. If unavailable, we fall back to standard HuggingFace Transformers + PEFT.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model, LoraConfig

UNSLOTH_AVAILABLE = False
try:
    from unsloth import FastLanguageModel
    UNSLOTH_AVAILABLE = True
    print('Unsloth loaded successfully.')
except Exception:
    print('Unsloth not available — using HuggingFace Transformers + PEFT instead.')

if UNSLOTH_AVAILABLE:
    MODEL_NAME = 'unsloth/Qwen2.5-1.5B-Instruct'
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=2048,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                        'gate_proj', 'up_proj', 'down_proj'],
        lora_alpha=16,
        lora_dropout=0,
        use_gradient_checkpointing='unsloth',
    )
else:
    MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map='auto'
    )
    lora_config = LoraConfig(
        r=16, lora_alpha=16,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
        lora_dropout=0, task_type='CAUSAL_LM'
    )
    model = get_peft_model(model, lora_config)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

print(f'Model loaded: {MODEL_NAME}')
model.print_trainable_parameters()

---
## 6. Generate Training Prompts from Environment States

We reset the environment 50 times across all 4 tasks and build a prompt dataset.

In [ ]:
from datasets import Dataset

prompt_env = DataCleaningEnv(seed=42)
prompts = []
task_ids = ['task_enterprise_orchestration', 'task_missing_values',
            'task_duplicate_handling', 'task_complex_validation']

for i in range(50):
    task_id = task_ids[i % len(task_ids)]
    obs = prompt_env.reset(task_id=task_id, seed=2000 + i)
    prompts.append(build_prompt(obs))

dataset = Dataset.from_dict({'prompt': prompts})
print(f'Created {len(dataset)} training prompts from environment states')
print(f'Example prompt length: {len(prompts[0])} chars')

---
## 7. Run GRPO Training

The RL optimization loop. For each prompt:
1. Generate 4 candidate completions
2. Score each via `environment_reward_function` (runs `env.step()`)
3. Update policy to prefer higher-reward actions

30 steps on a T4 takes approximately 15-25 minutes.

In [ ]:
from trl import GRPOTrainer, GRPOConfig

config = GRPOConfig(
    output_dir='/content/grpo_checkpoint',
    num_generations=4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    max_steps=30,
    logging_steps=5,
    save_strategy='no',
    max_completion_length=512,
    report_to=[],
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    config=config,
    train_dataset=dataset,
    reward_funcs=[environment_reward_function],
)

print('Starting GRPO training...')
result = trainer.train()
print(f'\nTraining complete.')
print(f'Final loss: {result.training_loss:.4f}')
print(f'Total steps: {result.global_step}')

---
## 8. Evaluate the Trained Model

Run the trained model on 12 fresh environment episodes (3 per task) and measure reward.

In [ ]:
import matplotlib.pyplot as plt

eval_env = DataCleaningEnv(seed=99)
eval_rewards = []

if UNSLOTH_AVAILABLE:
    FastLanguageModel.for_inference(model)
else:
    model.eval()

for i in range(12):
    task_id = task_ids[i % len(task_ids)]
    obs = eval_env.reset(task_id=task_id, seed=5000 + i)
    prompt = build_prompt(obs)

    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1536).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.7, do_sample=True)
    completion = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    reward = environment_reward_function([completion], [prompt])[0]
    eval_rewards.append(reward)
    print(f'Episode {i+1:2d} | {task_id:36s} | reward = {reward:.4f}')

mean_reward = sum(eval_rewards) / len(eval_rewards)

# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#22c55e' if r > 0 else '#ef4444' for r in eval_rewards]
axes[0].bar(range(1, len(eval_rewards)+1), eval_rewards, color=colors)
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Reward')
axes[0].set_title('Per-Episode Rewards (Trained Model)')
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

axes[1].bar(['Trained Model'], [mean_reward], color='#3b82f6', width=0.4)
axes[1].set_ylabel('Mean Reward')
axes[1].set_title(f'Mean Reward: {mean_reward:.4f}')
axes[1].set_ylim(-0.5, 1.0)

plt.tight_layout()
plt.savefig('artifacts/grpo_eval_results.svg')
plt.show()

print(f'\nMean evaluation reward: {mean_reward:.4f}')

---
## 9. Save Training Metrics

In [ ]:
from pathlib import Path

metrics = {
    'training_mode': 'grpo_rl_training',
    'model': MODEL_NAME,
    'unsloth_used': UNSLOTH_AVAILABLE,
    'trl_grpo': True,
    'num_prompts': len(dataset),
    'num_generations': 4,
    'max_steps': 30,
    'train_loss': float(result.training_loss),
    'eval_mean_reward': round(mean_reward, 4),
    'eval_rewards': [round(r, 4) for r in eval_rewards],
    'reward_function': 'environment_grounded (env.step + grader)',
}

Path('artifacts').mkdir(exist_ok=True)
Path('artifacts/grpo_training_metrics.json').write_text(json.dumps(metrics, indent=2))
print('Metrics saved to artifacts/grpo_training_metrics.json')
print(json.dumps(metrics, indent=2))